In [1]:
import pandas as pd
import xarray as xr
import numpy as np

from metpy.calc import wind_direction, wind_speed, wind_components
from metpy.units import units

data_dir = '../data_out/'
stats_dir = '../statistics/'

In [2]:
def get_df_from_nc(file, var):
    ds = xr.open_dataset(file)
    df = pd.DataFrame(ds[var].values, columns=ds[ds[var].dims[1]].values.astype(int), index=pd.to_datetime(ds.time,utc=True))
    return df

def resample_df(wspd, wdir, res):
    df = pd.DataFrame(index=wspd.index)

    if res is None:
        df['wdir'] = wdir
        df['wspd'] = wspd
    else:
        df['u'],df['v'] = wind_components(wspd.values * units('m/s'), wdir.values * units.deg)
        #df = df.resample(res, origin='end_day').mean()
        df = df.rolling(res, center=True, min_periods=min_peridos).mean()
        df['wdir'] = wind_direction(df['u'].values * units('m/s'), df['v'].values * units('m/s'))
        df['wspd'] = wind_speed(df['u'].values * units('m/s'), df['v'].values * units('m/s'))
    return df

def get_value_pairs(h_obs, h_icon, res):
    df_obs  = resample_df(df_wspd[h_obs], df_wdir[h_obs], res).add_suffix('_obs')
    df_icon = resample_df(df_i_wspd[h_icon], df_i_wdir[h_icon], res).add_suffix('_icon')

    return pd.concat([df_obs, df_icon], axis=1).dropna()

def get_stats(df, min_wspd=0.0, time_limits=None):
    if min_wspd > 0.0:
        df = df[df['wspd_icon'] >= min_wspd]  # only consider records with minimum observed wind speed

    if time_limits is not None:
        df = df.loc[time_limits[0]:time_limits[1]]

    df['d_wdir'] = pd.concat([abs(df['wdir_icon'] - df['wdir_obs']),
                              abs(df['wdir_icon'] - df['wdir_obs'] - 360),
                              abs(df['wdir_icon'] - df['wdir_obs'] + 360)], axis=1).min(axis=1)

    stats = {'n':len(df),
             'wdir_rmse': np.sqrt((df['d_wdir']**2).mean()).round(2),
             'wdir_rmse_r': (np.sqrt((df['d_wdir']**2).mean()) / 180 * 100).round(2),
             'wdir_mae': df['d_wdir'].mean().round(2),
             'wspd_rmse': np.sqrt(((df['wspd_icon'] - df['wspd_obs'])**2).mean()).round(2),
             'wspd_mae': abs(df['wspd_icon'] - df['wspd_obs']).mean().round(2),
             'wspd_mbe': (df['wspd_icon'] - df['wspd_obs']).mean().round(2)
             }
    return stats

## Script settings

In [3]:
# do scope specific settings
scope_name = 'HEFEX III'

file_obs  = f'obs/HEFEX3__Obs_WindRanger_L1__avg30min_20250807-20250831.nc'
file_icon = f'icon/HEFEX3__ICON_v370_2030_cid35704_75ml__avg30min_20250805-20250905.nc'

res = '3h'
min_peridos = 6

## Get data (ICON & Obs) and resample to daily

The data used for this figure is available at Zenodo:

HEFEX III: [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.19596272.svg)](https://doi.org/10.5281/zenodo.19596272)

In [4]:
df_wdir = get_df_from_nc(data_dir + file_obs, f'wdir')
df_wspd = get_df_from_nc(data_dir + file_obs, f'wspd')

df_i_wdir = get_df_from_nc(data_dir + file_icon, f'wdir').iloc[:,-15:]
df_i_wspd = get_df_from_nc(data_dir + file_icon, f'wspd').iloc[:,-15:]

In [5]:
print('Heights WR:', sorted(df_wdir.columns))
print('Heights ICON:', sorted(df_i_wdir.columns.astype(int)))

Heights WR: [7, 10, 15, 20, 30, 40, 50, 75, 100, 125, 150, 175, 200]
Heights ICON: [5, 15, 26, 39, 53, 68, 84, 100, 117, 134, 152, 170, 189, 207, 226]


In [6]:
# 1. Resample entire time frame to 3h
min_wspd = 0.0
time_limits = None #['2025-08-10', '2025-08-10']

df_all = pd.DataFrame({
    '15m' : get_stats(get_value_pairs(15,15,res), min_wspd=min_wspd, time_limits=time_limits),
    '100m': get_stats(get_value_pairs(100,100,res), min_wspd=min_wspd, time_limits=time_limits),
    '150m': get_stats(get_value_pairs(150,152,res), min_wspd=min_wspd, time_limits=time_limits),
    '200m': get_stats(get_value_pairs(200,207,res), min_wspd=min_wspd, time_limits=time_limits),
})

out_file = stats_dir + f'HEFEX3__WindRanger_ICON_stats__res{res}_full.csv'
df_all.to_csv(out_file)
print('Saved stats to', out_file)
df_all.round(2)

Saved stats to ../statistics/HEFEX3__WindRanger_ICON_stats__res3h_full.csv


,15m,100m,150m,200m
n,1062.00,886.00,824.00,735.00
wdir_rmse,65.35,74.19,77.67,82.71
wdir_rmse_r,36.31,41.22,43.15,45.95
wdir_mae,45.13,54.97,57.04,61.41
wspd_rmse,1.34,1.39,1.37,1.88
wspd_mae,1.02,1.01,1.01,1.31
wspd_mbe,-0.15,-0.30,-0.31,-0.44


In [7]:
# 1. Resample 2025-08-10 hourly
min_wspd = 0.0
time_limits = ['2025-08-10', '2025-08-10']

df_day = pd.DataFrame({
    '15m' : get_stats(get_value_pairs(15,15,res), min_wspd=min_wspd, time_limits=time_limits),
    '100m': get_stats(get_value_pairs(100,100,res), min_wspd=min_wspd, time_limits=time_limits),
    '150m': get_stats(get_value_pairs(150,152,res), min_wspd=min_wspd, time_limits=time_limits),
    '200m': get_stats(get_value_pairs(200,207,res), min_wspd=min_wspd, time_limits=time_limits),
})

out_file = stats_dir + f'HEFEX3__WindRanger_ICON_stats__res{res}_{time_limits[0].replace("-","")}.csv'
df_day.to_csv(out_file)
print('Saved stats to', out_file)
df_day.round(2)

Saved stats to ../statistics/HEFEX3__WindRanger_ICON_stats__res3h_20250810.csv


,15m,100m,150m,200m
n,48.00,48.00,48.00,48.00
wdir_rmse,81.91,74.93,59.89,56.45
wdir_rmse_r,45.51,41.63,33.27,31.36
wdir_mae,57.53,61.45,44.93,40.45
wspd_rmse,1.16,0.88,0.81,1.08
wspd_mae,0.90,0.77,0.66,0.88
wspd_mbe,-0.27,-0.54,-0.34,-0.29


## Construct LaTeX table

- with full time frame stats
- 2025-08-10 stats in brackets
- rename index for better display

In [8]:
# merge
df_merged = pd.DataFrame(index=df_all.index, columns=df_all.columns)

for idx in df_merged.index:
    if idx.startswith('wspd_'):
        decimals = 2
    else:
        decimals = 0

    for col in df_all.columns:
        df_merged.loc[idx, col] = (
            f"{df_all.loc[idx, col]:.{decimals}f} "
            f"({df_day.loc[idx, col]:.{decimals}f})"
        )

print(df_merged)

                       15m           100m           150m           200m
n                1062 (48)       886 (48)       824 (48)       735 (48)
wdir_rmse          65 (82)        74 (75)        78 (60)        83 (56)
wdir_rmse_r        36 (46)        41 (42)        43 (33)        46 (31)
wdir_mae           45 (58)        55 (61)        57 (45)        61 (40)
wspd_rmse      1.34 (1.16)    1.39 (0.88)    1.37 (0.81)    1.88 (1.08)
wspd_mae       1.02 (0.90)    1.01 (0.77)    1.01 (0.66)    1.31 (0.88)
wspd_mbe     -0.15 (-0.27)  -0.30 (-0.54)  -0.31 (-0.34)  -0.44 (-0.29)


In [9]:
latex_table = df_merged.to_latex()

latex_table = latex_table.replace(r'n ', r'\multicolumn{2}{r|}{number of paired observations (N)} ')

latex_table = latex_table.replace(r'wdir_rmse ',   r'Wind direction (°) & RMSE ')
latex_table = latex_table.replace(r'wdir_rmse_r ', r' & RMSE$_{\mathrm{rel}}$ (\%) ')
latex_table = latex_table.replace(r'wdir_mae ',    r' & MAE ')

latex_table = latex_table.replace(r'wspd_rmse ', r'Wind speed (m s$^{-1}$) & RMSE ')
latex_table = latex_table.replace(r'wspd_mae ',  r' & MAE ')
latex_table = latex_table.replace(r'wspd_mbe ',  r' & MBE ')

print(latex_table)

\begin{tabular}{lllll}
\toprule
 & 15m & 100m & 150m & 200m \\
\midrule
\multicolumn{2}{r|}{number of paired observations (N)} & 1062 (48) & 886 (48) & 824 (48) & 735 (48) \\
Wind direction (°) & RMSE & 65 (82) & 74 (75) & 78 (60) & 83 (56) \\
 & RMSE$_{\mathrm{rel}}$ (\%) & 36 (46) & 41 (42) & 43 (33) & 46 (31) \\
 & MAE & 45 (58) & 55 (61) & 57 (45) & 61 (40) \\
Wind speed (m s$^{-1}$) & RMSE & 1.34 (1.16) & 1.39 (0.88) & 1.37 (0.81) & 1.88 (1.08) \\
 & MAE & 1.02 (0.90) & 1.01 (0.77) & 1.01 (0.66) & 1.31 (0.88) \\
 & MBE & -0.15 (-0.27) & -0.30 (-0.54) & -0.31 (-0.34) & -0.44 (-0.29) \\
\bottomrule
\end{tabular}

